# TABULATE examples — Jupyter

Every `NN_<slug>.ggsql` scenario in [`ggsql-jupyter/examples/`](.) executed through the `ggsql` Jupyter kernel. Each scenario is a pair of cells (markdown header + ggsql query). Output is rendered HTML via the kernel's `text/html` MIME type — the same renderer the CLI uses.

Run all cells with **Run → Run All Cells**, or re-generate this notebook from disk with `python3 ggsql-jupyter/examples/build_notebook.py`.

## `01_minimal.ggsql`

```
-- The minimal TABULATE query: just the bare `TABULATE` keyword after a
-- SELECT renders every column from the result, in source order. No column
-- list, no FORMAT, no LABEL — the whole table is the default.
```

In [ ]:
SELECT
  'Acme'             AS company, 120000 AS revenue, 18  AS employees UNION ALL SELECT
  'Globex',                       98000,             12 UNION ALL SELECT
  'Initech',                      54000,              7 UNION ALL SELECT
  'Umbrella',                    210000,             24 UNION ALL SELECT
  'Stark Industries',            480000,             55
TABULATE

## `02_reorder.ggsql`

```
-- Listing column names explicitly after `TABULATE` selects (and reorders)
-- those columns. Compare with `01_minimal.ggsql`, which uses the bare
-- `TABULATE` form to take every column in source order.
```

In [ ]:
SELECT
  'Acme'             AS company, 120000 AS revenue, 18 AS employees UNION ALL SELECT
  'Globex',                       98000,             12 UNION ALL SELECT
  'Initech',                      54000,              7 UNION ALL SELECT
  'Umbrella',                    210000,             24 UNION ALL SELECT
  'Stark Industries',            480000,             55
TABULATE employees, company, revenue

## `03_hide.ggsql`

```
-- `FORMAT col SETTING hide => true` drops a column from the rendered table
-- without removing it from the underlying query.
```

In [ ]:
SELECT
  'Acme'             AS company, 120000 AS revenue, 18 AS employees, 'NA'   AS internal_id UNION ALL SELECT
  'Globex',                       98000,             12,                       'GBX-001'   UNION ALL SELECT
  'Initech',                      54000,              7,                       'INI-001'   UNION ALL SELECT
  'Umbrella',                    210000,             24,                       'UMB-001'   UNION ALL SELECT
  'Stark Industries',            480000,             55,                       'SI-001'
TABULATE company, revenue, employees, internal_id
FORMAT internal_id
  SETTING hide => true

## `04_sql_filter.ggsql`

```
-- TABULATE composes with the full SQL surface — WHERE, ORDER BY, LIMIT all work
-- because the SQL portion is executed unchanged by the reader before TABULATE
-- shapes the result.
```

In [ ]:
SELECT
  'Acme'             AS company, 120000 AS revenue UNION ALL SELECT
  'Globex',                       98000           UNION ALL SELECT
  'Initech',                      54000           UNION ALL SELECT
  'Umbrella',                    210000           UNION ALL SELECT
  'Stark Industries',            480000
ORDER BY revenue DESC
LIMIT 3
TABULATE company, revenue

## `05_stub.ggsql`

```
-- `FORMAT STUB <col>` designates a column as the row-label "stub".
-- It is rendered as the first column with a heavier right border, matching
-- gt's `rowname_col` styling.
```

In [ ]:
SELECT
  'Acme'             AS company, 120000 AS revenue, 18 AS employees UNION ALL SELECT
  'Globex',                       98000,             12              UNION ALL SELECT
  'Initech',                      54000,              7              UNION ALL SELECT
  'Umbrella',                    210000,             24              UNION ALL SELECT
  'Stark Industries',            480000,             55
TABULATE company, revenue, employees
FORMAT STUB company

## `06_title_subtitle.ggsql`

```
-- `LABEL title => '...', subtitle => '...'` adds a two-line header above
-- the table. Title is large; subtitle is smaller, both centred.
```

In [ ]:
SELECT
  'Acme'             AS company, 120000 AS revenue, 18 AS employees UNION ALL SELECT
  'Globex',                       98000,             12              UNION ALL SELECT
  'Initech',                      54000,              7              UNION ALL SELECT
  'Umbrella',                    210000,             24              UNION ALL SELECT
  'Stark Industries',            480000,             55
TABULATE company, revenue, employees
LABEL
  title    => 'Top customers, FY26',
  subtitle => 'Revenue and headcount snapshot'

## `07_column_labels.ggsql`

```
-- `LABEL <col> => '<str>'` overrides the column header without renaming
-- the underlying column. `LABEL caption => '...'` adds a source-note row
-- in the table footer.
```

In [ ]:
SELECT
  'Acme'             AS company, 120000 AS revenue, 18 AS employees UNION ALL SELECT
  'Globex',                       98000,             12              UNION ALL SELECT
  'Initech',                      54000,              7              UNION ALL SELECT
  'Umbrella',                    210000,             24              UNION ALL SELECT
  'Stark Industries',            480000,             55
TABULATE company, revenue, employees
LABEL
  company   => 'Customer',
  revenue   => 'Revenue (USD)',
  employees => 'Headcount',
  caption   => 'Source: internal CRM extract, 2026-Q2'

## `08_number_format.ggsql`

```
-- `FORMAT <cols> RENAMING * => '{:num %\'d}'` formats numbers with a
-- locale-aware thousands separator. The `\'` is a backslash-escaped
-- apostrophe inside a single-quoted string.
```

In [ ]:
SELECT
  'Acme'             AS company, 1200000 AS revenue, 1825 AS employees UNION ALL SELECT
  'Globex',                       980000,             1212               UNION ALL SELECT
  'Initech',                      540500,              704               UNION ALL SELECT
  'Umbrella',                   2100000,             2430                UNION ALL SELECT
  'Stark Industries',           4800500,             5512
TABULATE company, revenue, employees
FORMAT revenue, employees
  RENAMING * => '{:num %\'d}'

## `09_full_header.ggsql`

```
-- Phase 2 fully composed: stub column, title + subtitle, custom column
-- labels, caption (source note), and thousands-separator number formatting.
```

In [ ]:
SELECT
  'Acme'             AS company, 1200000 AS revenue, 1825 AS employees UNION ALL SELECT
  'Globex',                       980000,             1212               UNION ALL SELECT
  'Initech',                      540500,              704               UNION ALL SELECT
  'Umbrella',                   2100000,             2430                UNION ALL SELECT
  'Stark Industries',           4800500,             5512
TABULATE company, revenue, employees
FORMAT STUB company
FORMAT revenue, employees
  RENAMING * => '{:num %\'d}'
LABEL
  title     => 'Top customers, FY26',
  subtitle  => 'Revenue and headcount snapshot',
  revenue   => 'Revenue (USD)',
  employees => 'Headcount',
  caption   => 'Source: internal CRM extract, 2026-Q2'

## `10_spanner.ggsql`

```
-- `FORMAT SPAN <cols> AS <id>` groups columns under a shared header cell.
-- The bareword after `AS` is the spanner id; without a `LABEL <id> => ...`
-- override the id text is rendered verbatim.
```

In [ ]:
SELECT
  'Acme'             AS company, 120000 AS revenue, 18000 AS expenses, 18 AS employees UNION ALL SELECT
  'Globex',                       98000,             21000,             12              UNION ALL SELECT
  'Initech',                      54000,             19500,              7              UNION ALL SELECT
  'Umbrella',                    210000,             40000,             24              UNION ALL SELECT
  'Stark Industries',            480000,            120000,             55
TABULATE company, revenue, expenses, employees
FORMAT STUB company
FORMAT SPAN revenue, expenses AS financials

## `11_two_spanners.ggsql`

```
-- Two side-by-side spanners. `LABEL <id> => '<text>'` overrides the
-- spanner display text without changing the underlying column ids.
```

In [ ]:
SELECT
  'Toronto'  AS city, 2731571 AS population_2016, 2794356 AS population_2021,
                       4334.41 AS density_2016,    4427.75 AS density_2021 UNION ALL SELECT
  'Ottawa',          934243,                       1017449,
                       334.81,                      364.91                  UNION ALL SELECT
  'Mississauga',     721599,                        717961,
                      2467.93,                     2452.56                  UNION ALL SELECT
  'Brampton',        593638,                        656480,
                      2236.78,                     2468.99                  UNION ALL SELECT
  'Hamilton',        536917,                        569353,
                       490.10,                      518.91
TABULATE city, population_2016, population_2021, density_2016, density_2021
FORMAT SPAN population_2016, population_2021 AS population
FORMAT SPAN density_2016, density_2021 AS density
LABEL
  population => 'Population',
  density    => 'Density'

## `12_nested_spanners.ggsql`

```
-- Nested spanners: a later `FORMAT SPAN` can list earlier spanner ids as
-- its children, producing stacked header rows.
```

In [ ]:
SELECT
  'Toronto'  AS city, 2731571 AS population_2016, 2794356 AS population_2021,
                       4334.41 AS density_2016,    4427.75 AS density_2021 UNION ALL SELECT
  'Ottawa',          934243,                       1017449,
                       334.81,                      364.91                  UNION ALL SELECT
  'Mississauga',     721599,                        717961,
                      2467.93,                     2452.56                  UNION ALL SELECT
  'Brampton',        593638,                        656480,
                      2236.78,                     2468.99                  UNION ALL SELECT
  'Hamilton',        536917,                        569353,
                       490.10,                      518.91
TABULATE city, population_2016, population_2021, density_2016, density_2021
FORMAT SPAN population_2016, population_2021 AS population
FORMAT SPAN density_2016, density_2021       AS density
FORMAT SPAN population, density              AS comparison
LABEL
  population => 'Population',
  density    => 'Density',
  comparison => '2016–2021 Comparison'

## `13_full_spanner_report.ggsql`

```
-- Phase 1–3 fully composed: stub column, title / subtitle / caption,
-- per-column relabels, thousands-separator formatting, AND a nested
-- spanner block over the numeric columns.
```

In [ ]:
SELECT
  'Acme'             AS company,
  1200000            AS revenue_2025, 1825 AS employees_2025,
  1450000            AS revenue_2026, 1990 AS employees_2026 UNION ALL SELECT
  'Globex',           980000, 1212, 1100000, 1305            UNION ALL SELECT
  'Initech',          540500,  704,  610000,  742            UNION ALL SELECT
  'Umbrella',        2100000, 2430, 2380000, 2615            UNION ALL SELECT
  'Stark Industries',4800500, 5512, 5210000, 5901
TABULATE company, revenue_2025, employees_2025, revenue_2026, employees_2026
FORMAT STUB company
FORMAT SPAN revenue_2025, employees_2025 AS y2025
FORMAT SPAN revenue_2026, employees_2026 AS y2026
FORMAT SPAN y2025, y2026                 AS snapshot
FORMAT revenue_2025, revenue_2026, employees_2025, employees_2026
  RENAMING * => '{:num %\'d}'
LABEL
  revenue_2025   => 'Revenue (USD)',
  employees_2025 => 'Headcount',
  revenue_2026   => 'Revenue (USD)',
  employees_2026 => 'Headcount',
  y2025          => 'FY2025',
  y2026          => 'FY2026',
  snapshot       => 'Year-over-year snapshot',
  title          => 'Top customers, FY25 vs FY26',
  subtitle       => 'Revenue and headcount, side-by-side',
  caption        => 'Source: internal CRM extract, 2026-Q2'

## `14_widths_align.ggsql`

```
-- `FORMAT <col> SETTING width => '<css>'` fixes a column to a CSS width,
-- switching the table to `table-layout: fixed`. `SETTING align => '...'`
-- overrides the auto-aligned default ('left', 'right', or 'center').
```

In [ ]:
SELECT
  'Acme'             AS company, 1200000 AS revenue, 18 AS employees UNION ALL SELECT
  'Globex',                       980000,            12              UNION ALL SELECT
  'Initech',                      540500,             7              UNION ALL SELECT
  'Umbrella',                    2100000,            24              UNION ALL SELECT
  'Stark Industries',            4800500,            55
TABULATE company, revenue, employees
FORMAT company
  SETTING width => '200px'
FORMAT revenue
  SETTING width => '120px', align => 'right'
  RENAMING * => '{:num %\'d}'
FORMAT employees
  SETTING width => '100px', align => 'center'

## `15_align_override.ggsql`

```
-- `FORMAT <col> SETTING align => '<dir>'` overrides the automatic
-- alignment. Numeric columns default to right-aligned; here we force the
-- `score` column to center-aligned, and the text `team` column to
-- right-aligned to demonstrate that the override works in both directions.
```

In [ ]:
SELECT
  'Alpha'   AS team, 91 AS score UNION ALL SELECT
  'Beta',            74           UNION ALL SELECT
  'Gamma',           82           UNION ALL SELECT
  'Delta',           68           UNION ALL SELECT
  'Epsilon',         95
TABULATE team, score
FORMAT score
  SETTING align => 'center'
FORMAT team
  SETTING align => 'right'

## `16_widths_with_spanner.ggsql`

```
-- Widths compose with spanners. Each child column under a spanner can fix
-- its own CSS width via `FORMAT <col> SETTING width => '<css>'`, while
-- the spanner header sized to the sum of its children.
```

In [ ]:
SELECT
  'Acme'             AS company,
  1200000            AS rev_q1, 1450000 AS rev_q2,
  18                 AS hc_q1,  22      AS hc_q2 UNION ALL SELECT
  'Globex',           980000,            1020000,
                      12,                14               UNION ALL SELECT
  'Initech',          540500,             612000,
                       7,                  9              UNION ALL SELECT
  'Umbrella',        2100000,            2310000,
                      24,                28               UNION ALL SELECT
  'Stark Industries',4800500,            5120500,
                      55,                60
TABULATE company, rev_q1, rev_q2, hc_q1, hc_q2
FORMAT company
  SETTING width => '180px'
FORMAT rev_q1, rev_q2
  SETTING width => '110px'
  RENAMING * => '${:num %\'d}'
FORMAT hc_q1, hc_q2
  SETTING width => '70px', align => 'center'
FORMAT SPAN rev_q1, rev_q2 AS revenue
FORMAT SPAN hc_q1, hc_q2 AS headcount
LABEL
  revenue   => 'Revenue (USD)',
  headcount => 'Headcount',
  rev_q1    => 'Q1',
  rev_q2    => 'Q2',
  hc_q1     => 'Q1',
  hc_q2     => 'Q2'

## `17_num_decimals.ggsql`

```
-- `{:num %<printf>}` is the cell formatter mini-language. `.3f` fixes the
-- output to three digits after the decimal point.
```

In [ ]:
SELECT
  'alpha'  AS region, 0.1111 AS share UNION ALL SELECT
  'beta',             0.2225          UNION ALL SELECT
  'gamma',            0.3334          UNION ALL SELECT
  'delta',            0.5550          UNION ALL SELECT
  'epsilon',          5550.0
TABULATE region, share
FORMAT share
  RENAMING * => '{:num %.3f}'

## `18_num_thousands.ggsql`

```
-- Integer formatting with thousands separators: the `'` flag (printf
-- locale-aware grouping) inserts the locale-appropriate group separator.
-- Use `d` (no flag) for plain integers without separators.
-- The `\'` is a backslash-escaped apostrophe inside a single-quoted string.
```

In [ ]:
SELECT
  'Acme'             AS company, 1200000 AS revenue, 1825 AS employees UNION ALL SELECT
  'Globex',                       980000,             1212               UNION ALL SELECT
  'Initech',                      540500,              704               UNION ALL SELECT
  'Umbrella',                   2100000,             2430                UNION ALL SELECT
  'Stark Industries',           4800500,             5512
TABULATE company, revenue, employees
FORMAT revenue, employees
  RENAMING * => '{:num %\'d}'

## `19_currency.ggsql`

```
-- Literal prefix / suffix surrounds the `{:num %...}` template. A `$`
-- prefix yields currency-style output; mix and match with thousands
-- separators for ledgers.
```

In [ ]:
SELECT
  'Ford GT'            AS model, 2017 AS year,  647 AS hp, 447000 AS msrp UNION ALL SELECT
  'Ferrari 488 GTB',           2015,            661,        245400        UNION ALL SELECT
  'Lamborghini Huracan',       2015,            602,        237250        UNION ALL SELECT
  'Porsche 911 GT3',           2018,            500,        143600        UNION ALL SELECT
  'McLaren 570S',              2016,            562,        184900
TABULATE model, year, hp, msrp
FORMAT STUB model
FORMAT hp
  RENAMING * => '{:num %\'d}'
FORMAT msrp
  RENAMING * => '${:num %\'d}'

## `20_percent.ggsql`

```
-- A trailing `%` in the RENAMING template is just a literal character
-- tacked on to each formatted number — there is no scaling. If your
-- column holds 0-1 proportions and you want them displayed as
-- percentages, multiply by 100 in the SQL before TABULATE.
```

In [ ]:
SELECT month, pizzas_sold, pct_of_annual * 100 AS pct_of_annual FROM (
  VALUES
    ( 1,  9234, 0.0824),
    ( 2,  8543, 0.0762),
    ( 3,  9612, 0.0857),
    ( 4,  9412, 0.0839),
    ( 5,  9756, 0.0870),
    ( 6,  9234, 0.0823),
    ( 7, 10123, 0.0902),
    ( 8,  9854, 0.0879),
    ( 9,  9145, 0.0816),
    (10,  9534, 0.0850),
    (11,  9678, 0.0863),
    (12,  7034, 0.0627)
) AS sales(month, pizzas_sold, pct_of_annual)
TABULATE month, pizzas_sold, pct_of_annual
FORMAT STUB month
FORMAT pizzas_sold
  RENAMING * => '{:num %\'d}'
FORMAT pct_of_annual
  RENAMING * => '{:num %.1f}%'

## `21_scientific.ggsql`

```
-- Scientific notation: `{:num %.Ne}` renders the mantissa with N decimals
-- and emits an HTML `<sup>` exponent. The Unicode minus (U+2212) is used
-- for negative exponents to match gt's typography.
```

In [ ]:
SELECT
  0.000001     AS small, 1234567000.0    AS large UNION ALL SELECT
  0.00000345,            8765432.0                UNION ALL SELECT
  0.000000089,           543219876543.0           UNION ALL SELECT
  4.2,                   12.0
TABULATE small, large
FORMAT small, large
  RENAMING * => '{:num %.2e}'

## `22_dates.ggsql`

```
-- `{:time <strftime>}` formats DATE / TIME / TIMESTAMP values. Use the
-- standard strftime directives; `%-d` / `%-I` request pad-stripped output
-- (no leading zero).
```

In [ ]:
SELECT
  CAST('2015-01-15' AS DATE) AS date_launched, 'alpha'   AS phase UNION ALL SELECT
  CAST('2015-02-04' AS DATE),                   'beta'           UNION ALL SELECT
  CAST('2015-03-22' AS DATE),                   'rc1'            UNION ALL SELECT
  CAST('2015-04-30' AS DATE),                   'ga'             UNION ALL SELECT
  CAST('2015-06-12' AS DATE),                   'patch'
TABULATE date_launched, phase
FORMAT date_launched
  RENAMING * => '{:time %B %-d, %Y}'

## `23_datetime.ggsql`

```
-- Date, time, and datetime columns can all be formatted in one table,
-- each with its own `{:time ...}` template. Temporal columns
-- automatically right-align.
```

In [ ]:
SELECT
  CAST('2015-01-15' AS DATE)             AS the_date,
  CAST('13:35:00'   AS TIME)             AS the_time,
  CAST('2018-01-01 02:22:00' AS TIMESTAMP) AS the_datetime UNION ALL SELECT
  CAST('2015-06-04' AS DATE),
  CAST('08:05:00'   AS TIME),
  CAST('2018-07-04 14:30:00' AS TIMESTAMP)               UNION ALL SELECT
  CAST('2015-11-22' AS DATE),
  CAST('22:15:00'   AS TIME),
  CAST('2018-12-31 23:59:00' AS TIMESTAMP)
TABULATE the_date, the_time, the_datetime
FORMAT the_date
  RENAMING * => '{:time %-d %B %Y}'
FORMAT the_time
  RENAMING * => '{:time %-I:%M %p}'
FORMAT the_datetime
  RENAMING * => '{:time %A, %B %-d, %Y at %-I:%M %p}'

## `24_french_locale.ggsql`

```
-- Per-column locale: `FORMAT <col> SETTING locale => '<tag>'` switches
-- the month / weekday names used by `{:time ...}`. Currently `'en'`
-- (default) and `'fr'` are supported.
```

In [ ]:
SELECT
  CAST('2015-01-15' AS DATE) AS date_launched UNION ALL SELECT
  CAST('2015-02-04' AS DATE)                   UNION ALL SELECT
  CAST('2015-03-22' AS DATE)                   UNION ALL SELECT
  CAST('2015-04-30' AS DATE)                   UNION ALL SELECT
  CAST('2015-06-12' AS DATE)
TABULATE date_launched
FORMAT date_launched
  SETTING locale => 'fr'
  RENAMING * => '{:time %A %-d %B %Y}'

## `25_replace_missing.ggsql`

```
-- `RENAMING null => '<text>'` substitutes a fixed string for missing
-- values (gt's `sub_missing`). The replacement text is run through gt's
-- smart-text processor, so `---` renders as an em-dash (—) and `--` as an
-- en-dash (–).
```

In [ ]:
SELECT
  'Widget A' AS product, CAST(150  AS INTEGER) AS in_stock, CAST(  9.95 AS DOUBLE) AS price UNION ALL SELECT
  'Widget B',            CAST(NULL AS INTEGER),             CAST( 19.50 AS DOUBLE)          UNION ALL SELECT
  'Widget C',            CAST( 75  AS INTEGER),             CAST(  NULL AS DOUBLE)          UNION ALL SELECT
  'Widget D',            CAST(  0  AS INTEGER),             CAST(  4.25 AS DOUBLE)
TABULATE product, in_stock, price
FORMAT in_stock
  RENAMING null => 'N/A'
FORMAT price
  RENAMING null => '---'

## `26_replace_zero.ggsql`

```
-- `RENAMING 0 => '<text>'` substitutes a fixed string for cells whose
-- value is exactly zero (gt's `sub_zero`). Precedence: `0` wins over the
-- wildcard `*` formatter, so non-zero values still pass through
-- `{:num %\'d}`.
```

In [ ]:
SELECT
  'Widget A' AS product, CAST(150 AS INTEGER) AS in_stock, CAST(  0 AS INTEGER) AS on_order UNION ALL SELECT
  'Widget B',            CAST(  0 AS INTEGER),             CAST(200 AS INTEGER)             UNION ALL SELECT
  'Widget C',            CAST( 75 AS INTEGER),             CAST(  0 AS INTEGER)             UNION ALL SELECT
  'Widget D',            CAST(  3 AS INTEGER),             CAST( 12 AS INTEGER)
TABULATE product, in_stock, on_order
FORMAT in_stock, on_order
  RENAMING * => '{:num %\'d}'
FORMAT in_stock
  RENAMING 0 => 'Out of stock'
FORMAT on_order
  RENAMING 0 => '-'

## `27_direct_value_mapping.ggsql`

```
-- `RENAMING '<value>' => '<text>'` maps an exact cell value to a
-- replacement string (gt's `text_transform`). Multiple pairs in one
-- RENAMING block become a small lookup table; unmatched values render
-- as-is.
```

In [ ]:
SELECT
  'API'       AS feature, 'Y' AS available UNION ALL SELECT
  'Dashboard',            'N'              UNION ALL SELECT
  'Reports',              'Y'              UNION ALL SELECT
  'Settings',             'P'              UNION ALL SELECT
  'Audit log',            'N'
TABULATE feature, available
FORMAT available
  RENAMING 'Y' => '✓ Yes',
           'N' => '✗ No',
           'P' => '◐ Partial'

## `28_scale_named_palette.ggsql`

```
-- `SCALE background TO <palette>` colours cell backgrounds along a named
-- gt palette (here `viridis`). The domain is inferred from the column's
-- min/max; `target =>` selects which columns receive the colour fill.
```

In [ ]:
SELECT
  'Acme'             AS company,  127 AS units UNION ALL SELECT
  'Globex',                        98           UNION ALL SELECT
  'Initech',                       54           UNION ALL SELECT
  'Umbrella',                     210           UNION ALL SELECT
  'Stark Industries',             480           UNION ALL SELECT
  'Wayne Enterprises',            312
TABULATE company, units
SCALE background TO viridis
  SETTING target => (units)

## `29_scale_explicit_colors.ggsql`

```
-- `SCALE background TO ('<color>', '<color>', ...)` interpolates between
-- explicit colour stops in CIE Lab space. Two stops produce a smooth
-- two-colour gradient; any valid CSS colour (named or hex) works.
```

In [ ]:
SELECT company, margin * 100 AS margin FROM (
  VALUES
    ('Acme',              0.182),
    ('Globex',            0.094),
    ('Initech',           0.241),
    ('Umbrella',          0.057),
    ('Stark Industries',  0.318),
    ('Wayne Enterprises', 0.205)
) AS firms(company, margin)
TABULATE company, margin
FORMAT margin
  RENAMING * => '{:num %.1f}%'
SCALE background TO ('white', 'darkgreen')
  SETTING target => (margin)

## `30_scale_explicit_domain.ggsql`

```
-- `SCALE background FROM (lo, hi) TO <palette>` pins the colour scale to
-- a fixed numeric range instead of inferring it from the data. Values
-- outside the domain render with the neutral `na.color` (mid-grey),
-- which makes "above target" and "below target" cells visually distinct.
-- `target => (col, col, …)` applies the same scale to several columns.
```

In [ ]:
SELECT
  'Q1' AS quarter, 84 AS north, 67 AS south, 92 AS east, 71 AS west UNION ALL SELECT
  'Q2',            91,           78,           88,         85       UNION ALL SELECT
  'Q3',            76,           82,           95,         90       UNION ALL SELECT
  'Q4',            88,           94,          103,         87
TABULATE quarter, north, south, east, west
SCALE background FROM (60, 100) TO RdYlGn
  SETTING target => (north, south, east, west)

## `31_scale_log_transform.ggsql`

```
-- `VIA log10` warps the colour ramp through a log10 transform before
-- mapping into the palette, so values that span many orders of magnitude
-- (here population from ~50k to ~1.4B) get a useful range of colours
-- instead of all collapsing to one end of the gradient.
```

In [ ]:
SELECT
  'Liechtenstein' AS country,        39327 AS population UNION ALL SELECT
  'Iceland',                        372520                UNION ALL SELECT
  'Ireland',                       5033165                UNION ALL SELECT
  'Canada',                       38929902                UNION ALL SELECT
  'Germany',                      83196078                UNION ALL SELECT
  'United States',               331893745                UNION ALL SELECT
  'India',                      1407563842
TABULATE country, population
FORMAT population
  RENAMING * => '{:num %\'d}'
SCALE background FROM (0, 2000000000) TO ('white', 'darkblue') VIA log10
  SETTING target => (population)

## `32_highlight_failing_scores.ggsql`

```
-- `HIGHLIGHT <col> FILTER <SQL predicate> SETTING <key> => <value>, ...`
-- flags individual cells when the row matches the predicate. Here failing
-- scores (< 60) get a bold red foreground on the score column only — the
-- name and grade columns stay neutral.
```

In [ ]:
SELECT * FROM (VALUES
  ('Alice',  92, 'A'),
  ('Bob',    58, 'F'),
  ('Carla',  74, 'C'),
  ('Dan',    45, 'F'),
  ('Eve',    88, 'B'),
  ('Frank',  67, 'D')
) AS students(name, score, grade)
TABULATE name, score, grade
HIGHLIGHT score
  FILTER score < 60
  SETTING face => 'bold', color => 'red'

## `33_highlight_region_row.ggsql`

```
-- A HIGHLIGHT column list applies the same style to every listed column on
-- rows where the FILTER predicate is true. Here the entire row for the
-- "West" region picks up the soft-yellow background, while the row label
-- column (`region`) stays unhighlighted because it is not in the list.
```

In [ ]:
SELECT * FROM (VALUES
  ('North', 120000, 1200, 0.18),
  ('South',  98000,  980, 0.22),
  ('East',  145000, 1450, 0.15),
  ('West',  132000, 1320, 0.27)
) AS sales(region, revenue, units, margin)
TABULATE region, revenue, units, margin
HIGHLIGHT revenue, units, margin
  FILTER region = 'West'
  SETTING background => '#fff3cd'

## `34_highlight_up_down_days.ggsql`

```
-- Two HIGHLIGHT clauses with mutually exclusive predicates paint up-days
-- green and down-days red across the whole row. Multiple HIGHLIGHTs are
-- allowed; on conflict the later clause wins (gt's tab_style semantics),
-- which is irrelevant here because the predicates partition the rows.
```

In [ ]:
SELECT * FROM (VALUES
  (DATE '2024-12-02', 580.10, 595.40),
  (DATE '2024-12-03', 595.40, 591.20),
  (DATE '2024-12-04', 591.20, 605.85),
  (DATE '2024-12-05', 605.85, 598.05),
  (DATE '2024-12-06', 598.05, 612.30)
) AS ticks(date, open, close)
TABULATE date, open, close
FORMAT open, close
  RENAMING * => '${:num %.2f}'
HIGHLIGHT date, open, close
  FILTER close > open
  SETTING background => '#90EE90', face => 'bold'
HIGHLIGHT date, open, close
  FILTER close < open
  SETTING background => '#FFB6C1', face => 'bold'

## `35_facet_basic_grouping.ggsql`

```
-- `FACET <col>` groups body rows by the named column, emitting a heading
-- row before each group and dropping the column from the body. The
-- enclosing TABULATE column list doesn't need to mention the grouping
-- column.
```

In [ ]:
SELECT * FROM (VALUES
  ('Hardware', 'Widget',     45000, 450),
  ('Hardware', 'Gadget',     62000, 620),
  ('Software', 'Gizmo',      38000, 380),
  ('Software', 'Doohickey',  51000, 510)
) AS sales(category, product, revenue, units)
TABULATE product, revenue, units
FACET category

## `36_facet_summary_sum.ggsql`

```
-- `FACET <col> SETTING target => (cols), aggregate => ('agg')` adds a
-- summary row per group below the body rows. `target` lists the columns
-- the aggregate is applied to; non-target columns render as `—`. The
-- summary label defaults to the aggregate function's name.
```

In [ ]:
SELECT * FROM (VALUES
  ('Hardware', 'Widget',     45000, 450),
  ('Hardware', 'Gadget',     62000, 620),
  ('Software', 'Gizmo',      38000, 380),
  ('Software', 'Doohickey',  51000, 510)
) AS sales(category, product, revenue, units)
TABULATE product, revenue, units
FACET category
  SETTING target => (revenue, units),
          aggregate => ('sum')

## `37_facet_multi_aggregate.ggsql`

```
-- Multiple `aggregate` functions produce multiple summary rows per group;
-- `label` overrides the default labels. Available aggregates: `sum`,
-- `min`, `max`, `avg`, `median`, `sd`. `side => 'top'` places the
-- summary rows above the body rows; the default is `'bottom'`.
```

In [ ]:
SELECT * FROM (VALUES
  ('North', 'Q1',  120.5),
  ('North', 'Q2',  148.2),
  ('North', 'Q3',  132.7),
  ('North', 'Q4',  160.4),
  ('South', 'Q1',   95.3),
  ('South', 'Q2',  102.6),
  ('South', 'Q3',  118.9),
  ('South', 'Q4',  127.0)
) AS sales(region, quarter, amount)
TABULATE quarter, amount
FORMAT amount RENAMING * => '${:num %.2f}'
FACET region
  SETTING target => (amount),
          aggregate => ('min', 'max', 'avg'),
          label => ('Low', 'High', 'Average')

## `38_case_title.ggsql`

```
-- `{:title}` re-cases each cell to title case (first letter of every
-- whitespace-separated word in upper case, the rest lower case). Useful
-- when source text is screaming-snake / lower-snake / inconsistent.
-- The keyword is case-insensitive — `{:title}`, `{:Title}`, and
-- `{:TITLE}` all behave the same way, matching SQL.
```

In [ ]:
SELECT * FROM (VALUES
  ('north america', 'ACME widgets',        12.40),
  ('SOUTH america', 'globex international', 18.10),
  ('europe',        'INITECH inc',          9.75),
  ('Asia pacific',  'soylent CORP',        21.00)
) AS sales(region, customer, revenue)
TABULATE region, customer, revenue
FORMAT region, customer
  RENAMING * => '{:title}'

## `39_case_upper_lower.ggsql`

```
-- `{:upper}` and `{:lower}` force every character of a string cell to
-- one case. Useful for normalising ticker symbols, country codes, or
-- system identifiers that arrive in mixed case. The keywords are
-- case-insensitive — `{:upper}`, `{:UPPER}`, and `{:Upper}` are
-- equivalent, matching SQL.
```

In [ ]:
SELECT * FROM (VALUES
  ('aapl', 'United States', 195.42),
  ('msft', 'United States', 412.10),
  ('tsm',  'Taiwan',        148.85),
  ('asml', 'Netherlands',   985.30)
) AS holdings(ticker, country, price_usd)
TABULATE ticker, country, price_usd
FORMAT ticker
  RENAMING * => '{:upper}'
FORMAT country
  RENAMING * => '{:lower}'

## `40_unit_in_label.ggsql`

```
-- Inline superscripts (and subscripts) in LABEL text use a gt-style
-- `^N` / `_N` markup: `^2` renders as ², `_2` renders as ₂. The
-- bare-run N accepts an optional sign + one or more digits
-- (`^2`, `^-2`, `_10`). For anything richer -- letters, spaces,
-- punctuation -- use the braced form `^{ ... }` / `_{ ... }`.
--
-- This is the canonical way to annotate column headers with units;
-- there is no separate `SETTING units` key. See example 53 for the
-- full markup reference.
```

In [ ]:
SELECT * FROM (VALUES
  ('Tokyo',     2194.07, 6168.7),
  ('Osaka',      225.21, 12012.2),
  ('Yokohama',   437.38, 8534.5),
  ('Nagoya',     326.45, 7148.9)
) AS cities(name, land_area, pop_density)
TABULATE name, land_area, pop_density
FORMAT STUB name
FORMAT land_area
  RENAMING * => '{:num %\'.1f}'
FORMAT pop_density
  RENAMING * => '{:num %\'d}'
LABEL
  land_area   => 'Land Area km^2',
  pop_density => 'Pop Density people/km^2'

## `41_forced_sign_growth.ggsql`

```
-- `{:num %+.1f}` adds a `+` flag to the printf body: positives render
-- with a leading `+`, negatives with a Unicode minus (`−`, U+2212) so
-- positive and negative cells visually align. A trailing literal `%` is
-- just tacked on after formatting; if your data is in 0-1 proportions,
-- multiply by 100 in SQL so the rendered number reads as a percentage.
```

In [ ]:
SELECT region, growth_rate * 100 AS growth_rate FROM (VALUES
  ('North',    0.045),
  ('South',   -0.021),
  ('East',     0.083),
  ('West',    -0.067),
  ('Central',  0.012)
) AS growth(region, growth_rate)
TABULATE region, growth_rate
FORMAT growth_rate
  RENAMING * => '{:num %+.1f}%'

## `42_comprehensive_report.ggsql`

```
-- Comprehensive sales report combining every TABULATE feature:
-- a SQL CTE feeds quarterly aggregates into the table; the header has
-- a title and subtitle; revenue + units are grouped under a spanner;
-- each column gets its own number format (currency, integer with
-- thousands, percent); SCALE paints the satisfaction column on a
-- white-to-green ramp; HIGHLIGHT bolds and reddens any satisfaction
-- below 70%; FACET groups rows by region and appends per-region
-- sum/mean summary rows.
```

In [ ]:
WITH quarterly(region, quarter, revenue, units, satisfaction) AS (
  VALUES
  ('East',  'Q1',  62.4, 1878, 58),
  ('East',  'Q2', 127.1, 1483, 63),
  ('East',  'Q3', 108.5, 1226, 82),
  ('East',  'Q4', 185.9, 1444, 55),
  ('North', 'Q1', 196.7,  645, 78),
  ('North', 'Q2',  67.6, 2144, 61),
  ('North', 'Q3', 121.2, 1871, 62),
  ('North', 'Q4', 134.0,  696, 81),
  ('South', 'Q1', 185.6, 2039, 86),
  ('South', 'Q2',  70.8,  725, 78),
  ('South', 'Q3', 198.3, 1003, 64),
  ('South', 'Q4', 192.0, 2469, 59),
  ('West',  'Q1', 117.0, 1768, 63),
  ('West',  'Q2', 175.4, 1421, 92),
  ('West',  'Q3', 160.6, 1125, 92),
  ('West',  'Q4', 171.7,  761, 84)
)

TABULATE region, quarter, revenue, units, satisfaction FROM quarterly

FORMAT STUB quarter
FORMAT SPAN revenue, units AS financial

FORMAT revenue
  RENAMING * => '${:num %\'.1f}'
FORMAT units
  RENAMING * => '{:num %\'d}'
FORMAT satisfaction
  RENAMING * => '{:num %d}%'

FACET region
  SETTING target    => (revenue, units),
          aggregate => ('sum', 'avg'),
          side      => 'bottom'

SCALE background FROM (0, 100) TO ('white', 'green')
  SETTING target => (satisfaction)

HIGHLIGHT satisfaction
  FILTER  satisfaction < 70
  SETTING face  => 'bold',
          color => 'red'

LABEL
  title        => '2024 Sales Performance',
  subtitle     => 'By Region and Quarter',
  financial    => 'Financial',
  revenue      => 'Revenue',
  units        => 'Units Sold',
  satisfaction => 'CSAT Score'

## `43_raw_passthrough.ggsql`

```
-- `{}` is a raw passthrough — every cell renders as its default
-- formatted value, with any literal text outside the braces appended
-- verbatim. There is no type coercion or formatting change.
--
-- Here a `$` prefix and ` USD` suffix are tacked on around the cell's
-- default text rendering.
```

In [ ]:
SELECT * FROM (VALUES
  ('Q1', 'Bronze',   '12.5'),
  ('Q2', 'Silver',  '127.5'),
  ('Q3', 'Gold',    '1820.0'),
  ('Q4', 'Diamond', '24500.0')
) AS picks(quarter, tier, amount)
TABULATE quarter, tier, amount
FORMAT STUB quarter
FORMAT amount
  RENAMING * => '${} USD'

## `44_facet_groups_restrict.ggsql`

```
-- `groups => [list]` restricts which group values get summary rows.
-- Without it, every group gets all the summaries. With it, only the
-- listed groups do. Compare this with 36_facet_summary_sum.ggsql,
-- which puts a Sum row under every group.
```

In [ ]:
SELECT * FROM (VALUES
  ('North', 'Q1',  120),
  ('North', 'Q2',  148),
  ('South', 'Q1',   95),
  ('South', 'Q2',  103),
  ('East',  'Q1',   78),
  ('East',  'Q2',   84),
  ('West',  'Q1',  132),
  ('West',  'Q2',  146)
) AS sales(region, quarter, amount)
TABULATE quarter, amount
FACET region
  SETTING target    => (amount),
          aggregate => ('sum'),
          groups    => ['North', 'South'],
          label     => 'Sum'

## `45_facet_groups_error.ggsql`

```
-- This example MUST error: it lists 'Antarctica' in `groups => [...]`
-- but no row has that region. Run with:
--   ./target/debug/ggsql run examples/tabulate/45_facet_groups_error.ggsql
-- and the expected output is a clean diagnostic of the form:
--   Failed to execute TABULATE query: Parse error: FACET groups:
--     'Antarctica' is not a value of grouping column 'region'
```

In [ ]:
SELECT * FROM (VALUES
  ('North', 'Q1', 120),
  ('South', 'Q1',  95),
  ('East',  'Q1',  78),
  ('West',  'Q1', 132)
) AS sales(region, quarter, amount)
TABULATE quarter, amount
FACET region
  SETTING target    => (amount),
          aggregate => ('sum'),
          groups    => ['North', 'Antarctica'],
          label     => 'Sum'

## `46_highlight_size.ggsql`

```
-- `HIGHLIGHT … SETTING size => '<css length>'` bumps the cell's font
-- size when the FILTER matches. Useful for drawing the eye to a row
-- that needs attention without a colour change.
```

In [ ]:
SELECT * FROM (VALUES
  ('Alpha',    42),
  ('Bravo',    91),
  ('Charlie', 100),
  ('Delta',    37)
) AS scores(team, points)
TABULATE team, points
FORMAT STUB team
HIGHLIGHT team, points
  FILTER points >= 90
  SETTING size => '20px', face => 'bold'

## `47_highlight_transform.ggsql`

```
-- `HIGHLIGHT … SETTING transform => 'uppercase' | 'lowercase'` runs the
-- cell text through CSS `text-transform`. Useful for normalising
-- categorical labels without rewriting the data.
```

In [ ]:
SELECT * FROM (VALUES
  ('alpha',   'active'),
  ('bravo',   'archived'),
  ('charlie', 'active'),
  ('delta',   'archived')
) AS records(team, status)
TABULATE team, status
HIGHLIGHT team
  FILTER status = 'active'
  SETTING transform => 'uppercase', face => 'bold'

## `48_highlight_decoration.ggsql`

```
-- `HIGHLIGHT … SETTING decoration => 'line-through' | 'underline' |
-- 'overline'` applies a CSS `text-decoration` to matching cells.
-- Useful for striking through obsolete entries or underlining a key
-- value.
```

In [ ]:
SELECT * FROM (VALUES
  ('Alpha',   'open'),
  ('Bravo',   'closed'),
  ('Charlie', 'open'),
  ('Delta',   'closed')
) AS tickets(ticket, status)
TABULATE ticket, status
HIGHLIGHT ticket, status
  FILTER status = 'closed'
  SETTING decoration => 'line-through', color => '#888'

## `49_scale_foreground.ggsql`

```
-- `SCALE foreground` colours the cell *text* along a continuous ramp.
-- Same syntax as `SCALE background` (palette stops or named palette,
-- optional `FROM (lo, hi)` domain, optional `VIA log10` transform),
-- but writes to CSS `color` instead of `background-color`.
```

In [ ]:
SELECT * FROM (VALUES
  ('Alpha',    0.1),
  ('Bravo',    0.4),
  ('Charlie',  0.6),
  ('Delta',    0.9)
) AS rates(name, error_rate)
TABULATE name, error_rate
FORMAT STUB name
FORMAT error_rate RENAMING * => '{:num %.2f}'
SCALE foreground FROM (0, 1) TO ('black', 'red')
  SETTING target => (error_rate)

## `50_scale_size.ggsql`

```
-- `SCALE size FROM (lo, hi) TO ('<lo-px>', '<hi-px>')` interpolates a
-- font-size between two CSS pixel values across the domain. Larger
-- values render with bigger text. Useful for emphasising magnitude
-- without colour.
```

In [ ]:
SELECT * FROM (VALUES
  ('Bronze',     12),
  ('Silver',    127),
  ('Gold',     1820),
  ('Diamond', 24500)
) AS picks(tier, score)
TABULATE tier, score
FORMAT STUB tier
FORMAT score RENAMING * => '{:num %\'d}'
SCALE size FROM (0, 25000) TO ('12px', '28px')
  SETTING target => (score)

## `51_scale_opacity.ggsql`

```
-- `SCALE opacity FROM (lo, hi) TO ('<lo>', '<hi>')` modulates the
-- alpha channel on the cell's background colour, mirroring gt's
-- `cell_fill(alpha = ...)`. The opacity stops are numbers between 0
-- (transparent) and 1 (opaque), written as strings.
--
-- Compose with `SCALE background`: the background colour comes from
-- the first scale, the alpha from the second; the cell renders as
-- `background-color: rgba(r, g, b, a);`.
```

In [ ]:
SELECT * FROM (VALUES
  ('Item A',  10),
  ('Item B',  40),
  ('Item C',  70),
  ('Item D', 100)
) AS catalog(item, relevance)
TABULATE item, relevance
FORMAT STUB item
SCALE background TO ('#0070D0', '#0070D0')
  SETTING target => (relevance)
SCALE opacity FROM (0, 100) TO ('0.2', '1.0')
  SETTING target => (relevance)

## `52_format_wildcard.ggsql`

```
-- `FORMAT *` is a wildcard that applies the clause to every visible
-- column. Useful for table-wide null/zero substitutions and other
-- whole-table treatments. Here a single FORMAT * replaces every
-- missing cell with `---`, which gt's smart-text processor renders
-- as an em-dash (—). `--` likewise becomes an en-dash and `...`
-- becomes a horizontal ellipsis (see example 25 for the substitution
-- rules in detail).
```

In [ ]:
SELECT * FROM (VALUES
  ('Alpha', 100, 'on track'),
  ('Bravo', NULL::int, 'paused'),
  ('Charlie', 250, NULL::varchar),
  (NULL::varchar, 75, 'on track')
) AS rows(name, units, status)
TABULATE name, units, status
FORMAT *
  RENAMING null => '---'

## `53_label_markup.ggsql`

```
-- The LABEL clause runs its text through a small markup processor:
--
--   ^N    -> <sup>N</sup>    e.g. `km^2` renders as km²
--   _N    -> <sub>N</sub>    e.g. `H_2O` renders as H₂O
--
-- The bare-run N is an optional sign followed by one or more digits
-- (`^2`, `^-2`, `_10`). For any other content -- letters, spaces,
-- punctuation -- use the braced form:
--
--   ^{ ... }  /  _{ ... }    arbitrary content up to the matching `}`
--
-- The processor also runs the smart-text substitutions inherited
-- from gt's markdown processor: `---` becomes an em-dash, `--` an
-- en-dash, `...` a horizontal ellipsis.
```

In [ ]:
SELECT * FROM (VALUES
  ('CO2 emissions',     415.0),
  ('H2O concentration',   0.05),
  ('Earth (10^9 t)',      8.1)
) AS rows(metric, value)
TABULATE metric, value
FORMAT STUB metric
LABEL
  metric => 'Metric',
  value  => 'Value (10^9) --- per ^{day}'